# Text and NLP preprocessing

## Goal

This walkthrough uses a small synthetic review table to show text quality detection, normalization, stopword removal, tokenization, vocabulary extraction, and derived features. The text contains no credentials or external data.

In [1]:
import pandas as pd
from autoprepml import TextPrepML

reviews = pd.DataFrame({
    "review_id": range(1, 9),
    "review": [
        "Fast delivery and clear instructions.",
        "<p>Useful product, but the setup was slow.</p>",
        "Contact support@example.test for help at https://example.test/help",
        "Excellent quality and reliable results.",
        "ok",
        "Excellent quality and reliable results.",
        "The package arrived damaged and support was unhelpful.",
        None,
    ],
})
reviews

,review_id,review
0,1,Fast delivery and clear instructions.
1,2,"<p>Useful product, but the setup was slow.</p>"
2,3,Contact support@example.test for help at https...
3,4,Excellent quality and reliable results.
4,5,ok
5,6,Excellent quality and reliable results.
6,7,The package arrived damaged and support was un...
7,8,None


### Detect and normalize text quality issues

In [2]:
preparer = TextPrepML(reviews, text_column="review")
issues = preparer.detect_issues()
preparer.clean_text(
    lowercase=True, remove_urls=True, remove_emails=True,
    remove_html=True, remove_extra_spaces=True
)
preparer.remove_stopwords()
preparer.remove_duplicates()
preparer.filter_by_length(min_length=10, max_length=500)
print("quality_issues:", {key: value for key, value in issues.items() if value})
print("rows_after_filtering:", len(preparer.df))
preparer.df[["review_id", "review"]]

quality_issues: {'missing_text': 1, 'very_short': 1, 'contains_urls': 1, 'contains_emails': 1, 'contains_html': 1, 'avg_length': 40.42857142857143, 'median_length': 39.0}
rows_after_filtering: 5


,review_id,review
0,1,fast delivery clear instructions.
1,2,"useful product, setup slow."
2,3,contact help
3,4,excellent quality reliable results.
6,7,package arrived damaged support unhelpful.


### Extract reusable NLP features

In [3]:
preparer.extract_features()
preparer.tokenize(method="word")
preparer.detect_language()
vocabulary = preparer.get_vocabulary(top_n=8)
feature_columns = [column for column in preparer.df if column.startswith("review_")]
print("feature_columns:", feature_columns)
print("top_terms:", vocabulary)
print("languages:", preparer.df["review_language"].value_counts().to_dict())

feature_columns: ['review_id', 'review_length', 'review_word_count', 'review_upper_count', 'review_digit_count', 'review_special_char_count', 'review_avg_word_len', 'review_tokens', 'review_language']
top_terms: {'fast': 1, 'delivery': 1, 'clear': 1, 'instructions.': 1, 'useful': 1, 'product,': 1, 'setup': 1, 'slow.': 1}
languages: {'other': 5}


## Checks

The resulting table is ready for downstream vectorization or classification.

In [4]:
assert len(preparer.df) == 5
assert preparer.df["review"].str.len().min() >= 10
assert "review_tokens" in preparer.df
print("Text workflow checks passed.")

Text workflow checks passed.
